#### Group Fairness 모델
- FairGNN (Adversarial Learning): 대표
    - FairVGNN (Adversarial Learning)

- EDITS (Edge Rewiring): 대표
    - UGE (Edge Rewiring) : 제외
    - FairEdit (Edge Rewiring)
    - GEAR (Edge Rewiring)

- FairWalk (Rebalancing): 대표
    - CrossWalk (Rebalancing)

- NIFTY (Optimization with Regularization)

- FairGB

In [1]:
import pandas as pd

### Avg.

In [ ]:
df1 = pd.read_csv('./outputs/exp_fairgate.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2])

# baseline 제외
df_rank_v1 = df[df["model"] != "GCN"].copy()

# recidivism 제외시
df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy()

# 제외 아닐 시
# df_rank = df_rank_v1.copy()

# OOM / 학습불가 등이 문자열로 들어간 경우 숫자로 변환
for col in ["acc_mean", "roc_auc_mean", "dp_mean", "eo_mean"]:
    df_rank[col] = pd.to_numeric(df_rank[col], errors="coerce")

# dataset별 rank 계산
df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(ascending=False, method="average")
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(ascending=False, method="average")
df_rank["rank_DP"]  = df_rank.groupby("dataset")["dp_mean"].rank(ascending=True,  method="average")
df_rank["rank_EO"]  = df_rank.groupby("dataset")["eo_mean"].rank(ascending=True,  method="average")

# accuracy / fairness 평균 rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)

# 전체 dataset에 대한 모델별 평균 rank
summary = (
    df_rank.groupby("model")[["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
                              "rank_acc_avg", "rank_fair_avg"]]
    .mean()
    .sort_values("rank_acc_avg")
)

print(summary.round(3))

           rank_Acc  rank_AUC  rank_DP  rank_EO  rank_acc_avg  rank_fair_avg
model                                                                       
FairGate      2.625     1.500    3.000    2.250         2.062          2.625
FairGB        4.000     1.750    6.000    3.750         2.875          4.875
FairGNN       2.375     5.500    6.500    6.625         3.938          6.562
FairVGNN      5.188     3.000    4.875    4.500         4.094          4.688
FairGT        5.625     3.125    3.875    3.625         4.375          3.750
NIFTY         5.000     6.000    5.875    6.125         5.500          6.000
FairEdit      4.750     6.500    5.500    6.250         5.625          5.875
EDITS         8.000     5.333    9.000    7.667         6.667          8.333
CrossWalk     5.938     7.438    3.812    4.688         6.688          4.250
FairWalk      6.125     7.562    3.188    4.562         6.844          3.875


### ours

In [ ]:
file_name ='exp_fairgate'
datasets = ['pokec_z', 'pokec_n', 'german', 'credit', 'recidivism', 'nba', 'income']

df = pd.read_csv(f'./outputs/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean', 
    'roc_auc_std',
    'dp_mean', 
    'dp_std', 
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

# pockec_z: 0
# pokec_n: 1
# german: 2
# credit: 3
# recidivism: 4

# df[df.dataset == datasets[6]]
df

,model,dataset,acc_mean,acc_std,roc_auc_mean,roc_auc_std,dp_mean,dp_std,eo_mean,eo_std
0,FairGate,pokec_z,0.6917,0.0031,0.7482,0.0054,0.0100,0.0052,0.0085,0.0045
1,FairGate,pokec_n,0.6918,0.0060,0.7314,0.0053,0.0735,0.0343,0.0958,0.0389
2,FairGate,pokec_z_g,0.6893,0.0036,0.7542,0.0049,0.0487,0.0130,0.0100,0.0038
3,FairGate,pokec_n_g,0.6868,0.0054,0.7290,0.0071,0.0230,0.0103,0.0770,0.0096
4,FairGate,german,0.6576,0.0268,0.6194,0.0202,0.0708,0.0432,0.0609,0.0288
5,FairGate,income,0.8087,0.0038,0.7709,0.0077,0.0251,0.0054,0.0220,0.0092
6,FairGate,recidivism,0.8531,0.0071,0.8853,0.0050,0.0711,0.0058,0.0508,0.0062
7,FairGate,nba,0.7239,0.0216,0.7802,0.0032,0.0405,0.0204,0.0470,0.0446
8,FairGate,credit,0.7016,0.0029,0.7344,0.0019,0.0736,0.0033,0.0436,0.0050


In [3]:
file_name ='exp_pokec_gender'
datasets = ['pokec_z', 'pokec_n', 'german', 'credit', 'recidivism', 'nba', 'income']

df = pd.read_csv(f'./outputs/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 'acc_std', 'roc_auc_mean', 'roc_auc_std',
    'dp_mean', 'dp_std', 'eo_mean', 'eo_std', 
    'time_sec_mean', 'time_sec_std']]

# pockec_z: 0
# pokec_n: 1
# german: 2
# credit: 3
# recidivism: 4

# df[df.dataset == datasets[6]]
df

,model,dataset,acc_mean,acc_std,roc_auc_mean,roc_auc_std,dp_mean,dp_std,eo_mean,eo_std,time_sec_mean,time_sec_std
0,FairGate,pokec_z_g,0.6892,0.0056,0.7559,0.0032,0.0699,0.0293,0.0176,0.0184,40.08,15.4941
1,FairGate,pokec_n_g,0.6880,0.0050,0.7287,0.0067,0.0271,0.0080,0.0809,0.0082,39.00,0.7416
2,NIFTY,pokec_z_g,0.6531,0.0267,0.6523,0.0286,0.0396,0.0121,0.0599,0.0211,45.20,8.2985
3,NIFTY,pokec_n_g,0.6654,0.0131,0.6614,0.0130,0.0754,0.0204,0.1328,0.0203,56.12,2.0969
4,FairGNN,pokec_z_g,0.6761,0.0038,0.6809,0.0026,0.0592,0.0182,0.0766,0.0226,16.80,0.5431
5,FairGNN,pokec_n_g,0.6836,0.0053,0.6797,0.0047,0.0780,0.0071,0.1341,0.0123,16.80,0.4301
6,FairVGNN,pokec_z_g,0.6625,0.0052,0.7393,0.0048,0.0127,0.0115,0.0380,0.0116,1661.62,303.3205


### baselines

In [55]:
file_name ='exp_baselines'
datasets = ['pokec_z', 'pokec_n', 'german', 'credit', 'recidivism', 'nba', 'income']

df_v2 = pd.read_csv(f'./outputs/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean',
    'roc_auc_std',
    'dp_mean',
    'dp_std',
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

model_order = ['FairGNN', 'FairVGNN', 'EDITS', 'FairEdit', 
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

# filtered_df = df_v2[(df_v2.dataset == 'nba') 
#                     # & (df_v2.model = 'FairEdit')
#                     ].copy()

filtered_df = df_v2.copy()

filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df.sort_values('model')

# df_v2[df_v2.model == 'FairGT']

,model,dataset,acc_mean,acc_std,roc_auc_mean,roc_auc_std,dp_mean,dp_std,eo_mean,eo_std
67,FairGNN,pokec_n_g,0.6836,0.0053,0.6797,0.0047,0.0780,0.0071,0.1341,0.0123
66,FairGNN,pokec_z_g,0.6761,0.0038,0.6809,0.0026,0.0592,0.0182,0.0766,0.0226
16,FairGNN,recidivism,0.9220,0.0018,0.9091,0.0027,0.0773,0.0006,0.0377,0.0015
15,FairGNN,credit,0.7873,0.0029,0.6029,0.0070,0.0867,0.0399,0.0461,0.0363
14,FairGNN,german,0.6424,0.0513,0.5861,0.0495,0.2305,0.2477,0.2464,0.2321
...,...,...,...,...,...,...,...,...,...,...
61,FairGT,recidivism,0.8669,0.0063,0.8961,0.0068,0.0100,0.0038,0.0545,0.0312
62,FairGT,nba,0.6441,0.0134,0.6958,0.0376,0.0100,0.0107,0.0659,0.0322
54,FairGT,pokec_z,0.6418,0.0070,0.6992,0.0059,0.0515,0.0153,0.0231,0.0201
56,FairGT,pokec_z_g,0.6224,0.0246,0.6835,0.0118,0.0815,0.0203,0.0467,0.0162


### gcn

In [63]:
file_name ='exp_gcn'
df_ab = pd.read_csv(f'./outputs/{file_name}.csv')
# df_ab = df_ab[df_ab.dataset == 'recidivism']
df_ab = df_ab.drop(['stage_label', 'run', 'seed'], axis=1)

df_ab.groupby(['dataset', 'stage']).agg(['mean', 'std']).round(4)

acc         roc_auc              f1              dp  \
                    mean     std    mean     std    mean     std    mean   
dataset    stage                                                           
credit     A0     0.7468  0.0018  0.7423  0.0022  0.8283  0.0016  0.1436   
german     A0     0.6120  0.0049  0.6459  0.0082  0.6768  0.0063  0.4957   
income     A0     0.7994  0.0007  0.7868  0.0019  0.5302  0.0058  0.1841   
nba        A0     0.7371  0.0152  0.7803  0.0027  0.7652  0.0220  0.0396   
pokec_n    A0     0.6904  0.0028  0.7368  0.0015  0.6514  0.0042  0.1058   
pokec_n_g  A0     0.6719  0.0108  0.7282  0.0070  0.6637  0.0053  0.0136   
pokec_z    A0     0.6878  0.0024  0.7581  0.0012  0.7021  0.0031  0.0647   
pokec_z_g  A0     0.6907  0.0023  0.7566  0.0022  0.6940  0.0055  0.1070   
recidivism A0     0.8504  0.0038  0.8916  0.0012  0.7977  0.0039  0.0755   

                              eo         time_sec          
                     std    mean     std     mean     std  
dataset    stage                                           
credit     A0     0.0030  0.1200  0.0037    16.02  2.8700  
german     A0     0.0116  0.5334  0.0103     6.06  2.7646  
income     A0     0.0043  0.2586  0.0089    14.74  2.4172  
nba        A0     0.0118  0.0528  0.0254     9.42  1.5595  
pokec_n    A0     0.0081  0.1252  0.0165     7.34  0.2302  
pokec_n_g  A0     0.0091  0.0501  0.0148    13.20  3.5057  
pokec_z    A0     0.0101  0.0515  0.0097     8.34  0.5857  
pokec_z_g  A0     0.0088  0.0471  0.0066     8.70  0.7550  
recidivism A0     0.0030  0.0422  0.0034    10.80  2.6038